# Standard guided recon: one slice, every model, plus the guide

The one-shot (end-to-end) counterpart of `i2sb_recon_sampling.ipynb`, for the runs trained on
Torch: LPDSNet, LGGS and TGVN at each depth. Each run's measurement is built exactly as its
validation builds it (`scripts/eval_guided_recon.py`), on the same slice with the same added noise
(`VAL_NOISE`, default 0 = the scan as acquired). The last cell scores every model on the same val
slices and writes a per-slice table keyed by (volume, slice), to join against other runs or the
i2sb notebook's numbers.

In [ ]:
%matplotlib inline
import os, pathlib, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from scripts.eval_guided_recon import load_recon_run, val_subset, measure, reconstruct, guide_image
from sb.i2sb_recon import online_support
from training.metrics import compute_metrics

ALL_CELLS = ["lpdsnet_R8", "lggs_R8_g1", "tgvn_c12_R8_g1", "tgvn_c10_R8_g1", "tgvn_c8_R8_g1",
             "tgvn_c6_R8_g1"]
try:                                   # which models to show; a plain list without ipywidgets
    import ipywidgets as widgets
    CELLS = widgets.SelectMultiple(options=ALL_CELLS, value=tuple(ALL_CELLS), rows=6,
                                   description="models")
    display(CELLS)
except ImportError:
    CELLS = ALL_CELLS

In [ ]:
# ===================== SETTINGS -- pick the models above, then run the cells below ===============
RUN_ROOT = "trained_nets/guided_recon/knee/s3"   # s3 = grid noise; s3_noise-low for the new runs
CKPT = "net_bestval.ckpt"
SLICE = 0                  # index into the fixed val subset (slice 12 of each val volume)
VAL_NOISE = 0.0            # ADDED measurement noise std: 0 = the scan as acquired; None = the run's
SEED = 0
SUPPORT_THRESH = None      # e.g. 0.9: also score the recons masked to the operator's online-ESPIRiT
                           # support, as the i2sb validation does; None = as the nets output them

cells_ = list(CELLS if isinstance(CELLS, (list, tuple)) else CELLS.value)
runs = {}
for c in cells_:
    try:
        runs[c] = load_recon_run(os.path.join(RUN_ROOT, c), CKPT)
    except (FileNotFoundError, OSError) as e:
        print(f"{c}: skipped ({e})")
subsets = {c: val_subset(r["cfg"]) for c, r in runs.items()}
print(f"{len(runs)} runs from {RUN_ROOT}; {len(next(iter(subsets.values())))} val slices")


def vol_name(ds, i):
    base, j = ds.dataset, ds.indices[i]
    pi, sl = base.index[j]
    return os.path.splitext(os.path.basename(base.pairs[pi][1]))[0], (
        sl if sl is not None else base.start_slice)


def recon_one(c, i):
    m = measure(runs[c], subsets[c][i], noise=VAL_NOISE, seed=SEED + i)
    x = reconstruct(runs[c], m)
    if SUPPORT_THRESH is not None:
        sup = online_support(m["y"], m["mask"], m["acs_lines"], SUPPORT_THRESH,
                             **(runs[c]["cfg"]["mri"].get("online_smaps_kws") or {}))
        x = x * sup.to(x.real.dtype)
    return m, x

In [ ]:
out = {c: recon_one(c, SLICE) for c in runs}
first = next(iter(out.values()))[0]
gt, zf, g_img = first["gt"], first["zf"], guide_image(first)
mr = next(iter(runs.values()))["cfg"]["training"].get("metric_range", 1.0)
print("slice:", vol_name(next(iter(subsets.values())), SLICE))
rows = [("zero-filled", zf)] + [(c, x) for c, (_, x) in out.items()]
for k, v in rows:
    m = compute_metrics(gt.abs(), v.abs(), data_range=mr)
    print(f"{k:<18} PSNR {float(m['psnr']):6.2f}  SSIM {float(m['ssim']):.4f}  NRMSE {float(m['nrmse']):.4f}")

In [ ]:
shows = [("target", gt), ("guide (own scale)", g_img)] + rows
shows = [(k, v) for k, v in shows if v is not None]
g_ = gt[0, 0].abs().cpu()
vmax = float(g_.flatten().quantile(0.99))
fig, ax = plt.subplots(2, len(shows), figsize=(3.0 * len(shows), 7.6))
for j, (k, v) in enumerate(shows):
    img = v[0, 0].abs().cpu()
    own = k.startswith("guide")
    ax[0, j].imshow(img, cmap="gray", vmin=0,
                    vmax=float(img.flatten().quantile(0.99)) if own else vmax)
    ax[0, j].set_title(k, fontsize=9)
    if k not in ("target",) and not own:
        ax[1, j].imshow((img - g_).abs(), cmap="magma", vmin=0, vmax=0.2 * vmax)
        ax[1, j].set_title("|error| x5", fontsize=9)
    else:
        ax[1, j].axis("off")
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### Every model on the same val slices

`N_EVAL` slices of the fixed val subset (each its own measurement seed, the same for every model).
Writes the per-slice table to `OUT_CSV`; slices are keyed by (volume, slice), so tables from
another run set or cluster can be joined on those columns.

In [ ]:
N_EVAL = 20
OUT_CSV = f"results/guided_recon_eval_{os.path.basename(RUN_ROOT)}.csv"

recs = []
n = min(N_EVAL, min(len(s) for s in subsets.values()))
for i in range(n):
    for c in runs:
        m, x = recon_one(c, i)
        vol, sl = vol_name(subsets[c], i)
        s = compute_metrics(m["gt"].abs(), x.abs(), data_range=mr)
        recs.append(dict(model=c, volume=vol, slice=int(sl), psnr=float(s["psnr"]),
                         ssim=float(s["ssim"]), nrmse=float(s["nrmse"])))
df = pd.DataFrame(recs)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f"{n} slices -> {OUT_CSV}")
df.groupby("model")[["psnr", "ssim", "nrmse"]].agg(["mean", "std"]).round(4)